# 스태킹(Stacking) — 모델들의 예측을 다시 학습하는 모델

수업 중 작성한 필기 `앙상블분류실습` · `앙상불회귀실습01` 의 스태킹 부분을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `머신러닝` p.108~113 — 스태킹 · 앙상블 회귀 · 모델 선택 기준)

- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.
- 그림 · **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 미리 만들어 두었습니다.

> 실습 데이터 `house_price.csv` · `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다.

In [ ]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 스태킹의 원리 (교재 p.108)

단순 투표 · 평균 대신, **개별 모델들의 예측 결과를 새로운 특성(Feature)으로** 만들어 **메타 모델(Meta-Learner)** 을 학습시키는 앙상블.

| | 1단계 (Base Models) | 2단계 (Meta Model / Blender) |
|---|---|---|
| **학습** | 서로 다른 알고리즘으로 각각 학습 | 1단계 **예측값 + 정답**으로 메타 모델 학습 |
| **예측** | 평가용 데이터를 각각 예측 | 1단계 **예측값을 입력(X)** 으로 최종 예측 |

![스태킹 흐름 애니메이션](./images/anim_04_stacking_flow.gif)

### 왜 교차 검증(cv)이 필요할까?

1단계 모델이 **학습에 쓴 데이터를 그대로 예측**하면 정답을 외운 상태라 예측값이 거의 정답과 같다.
메타 모델이 그걸 보고 배우면 "이 모델은 항상 맞힌다" 고 착각한다.
그래서 학습용을 5조각으로 나눠 **"자기가 안 본 조각"만 예측**하게 해서(`cv=5`) 메타 모델의 학습 데이터를 만든다.

## 2. 직접 만들어 보기 — `cross_val_predict` (보스턴 집값)

`StackingRegressor` 가 속에서 하는 일을 손으로 따라 해 보고, 결과가 같은지 확인한다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error as rmse
from xgboost import XGBRegressor

df_h = pd.read_csv('./house_price.csv')
X_h = df_h.drop(columns=['medv'])
y_h = df_h['medv']
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.25, random_state=0)

# 1단계 기본 모델 3개
base_models = {
    'dt': DecisionTreeRegressor(random_state=0),
    'rf': RandomForestRegressor(random_state=0),
    'xgb': XGBRegressor(random_state=0),
}

# ① 학습용에서 '안 본 조각' 예측(cv=5) → 메타 모델의 학습용 X
meta_X_train = pd.DataFrame({name: cross_val_predict(model, X_train_h, y_train_h, cv=5)
                             for name, model in base_models.items()})

# ② 기본 모델들은 학습용 전체로 다시 학습 → 평가용 예측 → 메타 모델의 평가용 X
meta_X_test = pd.DataFrame({name: model.fit(X_train_h, y_train_h).predict(X_test_h)
                            for name, model in base_models.items()})

print("메타 모델의 학습용 입력 (앞 5행) :")
print(meta_X_train.head().round(2))

In [ ]:
# ③ 2단계 : 메타 모델 학습 → 최종 예측
meta = LinearRegression().fit(meta_X_train, y_train_h)
pred_manual = meta.predict(meta_X_test)

print("메타 모델 가중치 :", ", ".join(f"{n} {c:.3f}" for n, c in zip(meta_X_train.columns, meta.coef_)))
print(f"직접 만든 스태킹 RMSE : {rmse(y_test_h, pred_manual):.4f}")

## 3. 스태킹 회귀 — `StackingRegressor` (교재 p.112)

In [ ]:
from sklearn.ensemble import StackingRegressor

base_estimators = [
    ('dt', DecisionTreeRegressor(random_state=0)),
    ('rf', RandomForestRegressor(random_state=0)),
    ('xgb', XGBRegressor(random_state=0)),
]
final_meta_learner = LinearRegression()          # 회귀의 메타 모델

stack_reg = StackingRegressor(estimators=base_estimators, final_estimator=final_meta_learner,
                              cv=5, n_jobs=-1)
stack_reg.fit(X_train_h, y_train_h)
pred_stack_reg = stack_reg.predict(X_test_h)

print(f"StackingRegressor RMSE : {rmse(y_test_h, pred_stack_reg):.4f}")
print("메타 모델 가중치 :", ", ".join(f"{n} {c:.3f}" for (n, _), c in zip(base_estimators, stack_reg.final_estimator_.coef_)))
print("직접 만든 것과 같은가? :", np.allclose(pred_stack_reg, pred_manual))

> - 직접 만든 스태킹과 `StackingRegressor` 의 예측이 **똑같다** — 속에서 하는 일이 정확히 위의 ①②③ 이다
> - RMSE **4.20** — 기본 모델 중 가장 좋은 랜덤 포레스트(4.07)보다 조금 나쁘다
> - 메타 모델은 **xgb 0.854** · rf 0.098 · dt 0.079 로 XGBoost 를 가장 믿었다. 그런데 XGBoost 기본값은 학습용을 외우는 경향(학습용 RMSE 0.01)이 있어서 그 버릇까지 같이 따라갔다

## 4. 스태킹 분류 — `StackingClassifier` (교재 p.109~111)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

df = pd.read_csv('./diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
X_data = df.drop(columns=['Diabetes_binary'])
y_data = df['Diabetes_binary']
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=0, stratify=y_data
)

# 1단계 : 개별 기본 모델 — (이름, 모델) 튜플의 리스트
base_estimators = [
    ('rf', RandomForestClassifier(random_state=0)),
    ('xgb', XGBClassifier(random_state=0)),
    ('dt', DecisionTreeClassifier(random_state=0)),
]

# 2단계 : 최종 메타 모델 (Meta Learner / Blender)
final_meta_learner = LogisticRegression(max_iter=1000, random_state=0)

stack = StackingClassifier(estimators=base_estimators, final_estimator=final_meta_learner,
                           cv=5, n_jobs=-1)     # n_jobs=-1 : CPU 코어를 모두 사용

In [ ]:
# 학습 : 기본 모델 3개 × 교차 검증 5회 + 전체 재학습 → 약 20~30초
stack.fit(X_train, y_train)

pred_test_stack = stack.predict(X_test)
acc_stack = accuracy_score(y_test, pred_test_stack)
print(f"Stacking 평가용 정확도 : {acc_stack:.4f}")
print("메타 모델 가중치 :", ", ".join(f"{n} {c:.2f}" for (n, _), c in zip(base_estimators, stack.final_estimator_.coef_[0])))

In [ ]:
# 기본 모델 각각과 비교 (스태킹 안에서 학습된 모델을 꺼내 쓴다)
rows = []
for name, model in stack.named_estimators_.items():
    pred = model.predict(X_test)
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred)})
rows.append({'model': 'stacking', 'accuracy': acc_stack, 'f1': f1_score(y_test, pred_test_stack)})

result = pd.DataFrame(rows).set_index('model').sort_values('accuracy', ascending=False).round(4)
result

> - 스태킹 **74.8%** — RF(74.0%)보다 높고 XGBoost(75.0%)보다 조금 낮다
> - 메타 모델 가중치 **RF 1.63 · XGB 3.27 · DT −0.01** → 단일 결정 트리(65.5%)는 **사실상 무시**했다. 메타 모델이 "어느 모델을 얼마나 믿을지" 를 데이터로 배운 것
> - 스태킹은 **기본 모델들이 서로 다른 실수를 할 때** 효과가 크다. 오늘은 RF · XGB 가 둘 다 트리 앙상블이라 실수가 비슷했다

## 5. 모델 비교 · 선택 기준 (교재 p.113)

![모델 비교](./images/05_model_comparison.png)

| 단계 | 내용 |
|------|------|
| **1단계 비즈니스 제약 확인** | 설명 가능성이 필수인가? (→ 단일 트리 · 선형 모델) 추세 예측인가? |
| **2단계 후보 모델 일괄 실험** | 선형 / 로지스틱, SVM, 단일 트리, 랜덤 포레스트, 부스팅, 스태킹 |
| **3단계 정량 지표 평가 & 선택** | 분류 : **F1-score** · 회귀 : **RMSE** |

> **F1-score** 는 정밀도(예측한 당뇨 중 진짜 당뇨 비율)와 재현율(진짜 당뇨 중 찾아낸 비율)의 조화 평균이다.
> 이 데이터는 정답이 50 : 50 이라 정확도와 F1 의 순위가 같았다 (XGB 0.757 > Stacking 0.754 > RF 0.748).

> **📝 고친 부분**
> - 변수 이름 오타 **`final_mate_learmer`** → `final_meta_learner`
> - `print("평가용 데이터:", y_train.index)` → `y_train` 은 **학습용** 정답
> - `print(acc_rf)` · `print(acc_xgb)` … 숫자만 나열하던 결과 → 모델별 비교표(정확도 + F1)
> - `"예측 값:"` 으로 정확도를 찍던 문구 → 무엇을 출력하는지 맞게
> - 스태킹 회귀에 `train_test_split` `random_state` 고정 (다른 모델과 같은 분할)
> - 스태킹 내부 동작(`cross_val_predict`)을 직접 구현해 결과가 같은지 확인하는 코드, 메타 모델 가중치 · F1 비교 추가

## 6. 정리

| | 배깅 | 부스팅 | 스태킹 |
|---|---|---|---|
| 모으는 방법 | 투표 / 평균 | 잔차를 순서대로 보완 | **메타 모델이 학습** |
| 당뇨병 정확도 | 74.0% | **75.0%** | 74.8% |
| 보스턴 RMSE | 4.07 | 4.23 (깊이 3 : **3.76**) | 4.20 |

- 스태킹 = 1단계 모델들의 **예측값**을 2단계 메타 모델의 **입력**으로
- 메타 모델 학습용 예측은 **교차 검증(cv)** 으로 만든다 (외운 답 방지)
- **스태킹이 항상 1등은 아니다** — 기본 모델들이 서로 달라야 효과가 크다